### gensim 라이브러리 모델에서 GrodSearchCV를 이용하는 방법
- sklearn에서 제공한는 GridSearchCV 모델의 fit(), 'transform(), predict() 함수를 이용하여 최적의 매개변수의 값들을 찾는 class
- gensim의 모델들은 해당 함수들이 존재하지 않는다.
- class를 새로 생성하여 fit(), transform(), predict()함수를 내장한 객체를 생성
- pipeline, kfold, gridsearch를 이용

In [176]:
# 라이브러리 로드
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from gensim.models import FastText
from konlpy.tag import Komoran
import numpy as np

In [177]:
# Komoran 형태소 분석기를 이용한 토큰화 함수 정의
komoran = Komoran()
def tokenize(text):
    tokens =  []
    for word in text:
        return komoran.morphs(text)

In [207]:
# FastText를 fit(), transform()를 내장한 class 정의
class FastTExtVectorizer(BaseEstimator, TransformerMixin):
    # 생성자 함수 -> class가 생성될때 최초로 한번 실행이 되는 함수
    # class 내부에서 사용할 데이터들을 저장하기 위해 사용
    def __init__(
        self,
        # FastText에서 사용할 매개변수
        # 토큰화된 데이터 2차원 리스트
        # vector_size: (좌표축의 개수(차원의 수))
        # window : 주변의 체크할 단어의 개수
        # min_count : 최소 등장 획수
        # epochs : 반복 학습의 회수
        # sg : 확률 계산 방식 (주변 단어를 찾을 것인가 중심 단어를 찾을 것인가)
        # min_n : subword의 최소 개수(글자 수)
        # max_n : subword의 최대 개수(글자 수)
        # worker : 병렬 처리 스레드의 수
        tokenize = None,
        vector_size = 100,
        min_count = 1,
        window = 3, 
        sg = 1,
        epochs = 10,
        min_n = 3,
        max_n =6,
        worker=1
    ):
        # class 안에 저장공간을 생성하여 데이터들을 저장
        self.tokenize = tokenize
        self.vector_size = vector_size
        self.window = window
        self.sg =sg
        self.min_count = min_count
        self. max_n = max_n
        self.min_n = min_n
        self.epochs = epochs
        self.worker = worker
        # 시드값 고정
        self.seed = 42
        
        # 모델일아 단어 사전이 저장될 빈 공간을 생성
        self.model = None
        self.voca = None
        
    # 총 4개의 메서드를 생성
    # 토큰화 메서드
    def to_token(self, X):
        # 만약에 토큰화 함수가 존재하지 않는다면 공백을 기준으로 문자를 나눠준다.
        if self.tokenize is None:
            # s -> 문장 하나
            # t -> 문장 안에서 단어
            return [[t for t in s.split()]for s in X]
        return [self.tokenize(s) for s in X]
    
    # 학습 함수 -> fit(X, Y)
    def fit(self, X, Y):
        # class 내부의 to_token() 함수를 호출
        sentences = self.to_token(X)
        # FastText를 이용하여 sentences들을 학습
        self.model = FastText(
            sentences= sentences,
            vector_size=self.vector_size,
            window = self.window,
            min_count= self.min_count,
            sg = self.sg,
            epochs=self.epochs,
            min_n = self.min_n,
            max_n = self.max_n,
            workers=self.worker,
            seed = self.seed
        )
        # 학습 한 후 학습 데이터에서의 단어 사전 생성
        self.voca = set(self.model.wv.key_to_index.keys())
        return self
    
    # 문장에서 평균 벡터로 사용할 doc_vec() 함수를 정의
    def doc_vec(self, tokens):
        # 하나의 문장을 벡터화 -> 단위 벡터의 평균
        # self.model -> FastText 모델
        # wv -> 단어들과 단위 벡터들이 모여있는 dict 형태의 데이터
        vecs = [self.model.wv[w] for w in tokens if w in self.model.wv]
        if not vecs:
            # vecs가 존재하지 않은 경우 -> 빈 리스트인 경우
            # 희소행렬을 되돌려준다.
            v = np.zeros(self.model.vector_size)
        else:
            v = np.mean(vecs, axis=0)
            
        v /= (np.linalg.norm(v) + 1e-12)
        return v
    # 변형함수 정의 (transform 함수)
    def transform(self, X):
        # 토큰화
        sentences = self.to_token(X)
        # 임베딩
        mat = np.vstack([self.doc_vec(tokens) for tokens in sentences])
        return mat

In [200]:
# 1차 class 테스트
X = [
    '상품 품질이 아주 좋다',
    '배송이 너무 느리다',
    '포장이 잘되어 있고 만족합니다',
    '환불이 오래 걸려서 불안하다',
    '가격 대비 만족스럽다',
    '연락이 잘 안 되고 불친절하다',
    '디자인이 이쁘고 마음에 든다',
    '설명과 달라서 실망이다',
    '베송이 빠르고 기사분이 친절했습니다',
    '색상이 사진과 달라서 만족하지 못했습니다'
]

Y = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

In [201]:
vec_data = FastTExtVectorizer(tokenize=tokenize)

In [181]:
vec_data.fit(X, Y)

,tokenize,<function tok...0026B66715940>
,vector_size,100
,min_count,1
,window,3
,sg,1
,epochs,10
,min_n,3
,max_n,6
,worker,1


In [182]:
# transform 함수 테스트
vec_data.transform(X)

array([[-1.0718674e-03,  1.5845429e-04, -4.7648148e-04,  7.1088673e-04,
         1.7170577e-04,  1.6653448e-03, -1.2871929e-03,  3.4744432e-03,
        -7.0816994e-04, -1.3503934e-03,  1.7535685e-03, -1.2818743e-03,
         2.7966339e-03,  9.1702132e-05,  2.9037980e-04,  5.2294350e-04,
        -8.0424407e-04, -2.0848515e-03,  2.7646690e-03, -3.2827628e-04,
         2.4561517e-04, -1.3646459e-03,  9.3347911e-04,  3.8891428e-03,
        -9.5771007e-08, -8.9646183e-04, -2.9608253e-03,  1.6266169e-03,
        -1.0168809e-03, -5.9717544e-04, -2.3062488e-03, -9.6085580e-04,
        -1.9900282e-03,  4.7105015e-04,  1.5318204e-03, -1.8928139e-03,
         4.5889930e-04, -5.6768797e-04,  1.6356849e-04,  1.8968127e-03,
         1.2132470e-03, -1.9073021e-03, -1.1742893e-04,  7.4224727e-04,
         3.0625184e-04,  2.6189957e-03, -1.5637428e-03,  4.7447797e-04,
         1.2284945e-03,  5.6748302e-04,  1.8687649e-03, -1.9203468e-03,
         2.0061038e-04, -2.0450822e-03, -1.9384517e-03,  2.82514

In [208]:
# 파이프라인 생성 -> FastTextVectorizer객체와 머신러닝 모델 합친다.

pipe = Pipeline(
    [
        ('ft', FastTExtVectorizer(tokenize=tokenize)),
        ('clf', LogisticRegression(max_iter=2000))
    ]
)

In [184]:
pipe.fit(X, Y)

,steps,"[('ft', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,tokenize,<function tok...0026B66715940>
,vector_size,100
,min_count,1
,window,3
,sg,1
,epochs,10
,min_n,3


In [185]:
# 최적의 파라미터를 찾기 위해 GridSearch와 Fold화
cv = StratifiedKFold(n_splits=5 ,shuffle=True, random_state=42)

In [186]:
# 매개변수의 경우들을 dict 데이터로 생성
grid_params = {
    'ft__vector_size' : [50, 100],
    'ft__sg' : [0, 1],
    'clf__C' : [1.0, 2.0]
}

In [187]:
# pipe와 cv, grid_params들을 이용하여 GridSearchCV를 사용
grid = GridSearchCV(
    estimator=pipe,
    param_grid=grid_params,
    cv = cv,
    scoring='f1_macro', 
    verbose=1
)

In [188]:
# gridSearch 실행
grid.fit(X ,Y)
# 최적의 파라미터를 출력
print("GridSearch 기준 최적의 파라미터 : ", grid.best_params_)
print('GridSearch의 최적의 F1', grid.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
GridSearch 기준 최적의 파라미터 :  {'clf__C': 1.0, 'ft__sg': 0, 'ft__vector_size': 50}
GridSearch의 최적의 F1 0.39999999999999997


### 연습
- FastTextVectorizer, StratifiedKfold, pipeline, GridSearchCV를 이용하여 data 폴더 안에 ratings_train.txt 파일의 데이터를 학습하여 최적의 파라미터를 구하라
    1. train 데이터를 ratings_train.txt에서 상위 500개의 데이터를 이용
    2. text 데이터는 하위 100개의 데이터
    3. 파라미터 경우의 수
        - FastText
            - womdow -> 3, 5
            - sg -> 0, 1
            - epochs -> 10, 50
        - Logistic
            - C -> 1.0, 2.0
    4. Fold는 3개를 생성
    5. 최적의 파라미터를 생성
    6. 최적의 모델을 이용하여 test 데이터를 예측한 뒤 class_reprot 생성

In [209]:
import pandas as pd
from sklearn.metrics import classification_report

In [222]:
data = pd.read_csv('../data/ratings_train.txt', sep='\t')
data.head()
df = data.copy()

In [211]:
train = data[:500]
test = data[-100:]

In [212]:
X_train = train['document'].values
Y_train = train['label'].values

X_test = test['document'].values
Y_test = test['label'].values

In [216]:
grid_params2 = {
    'ft__window' : [3, 5],
    'ft__sg' : [0, 1],
    'ft__epochs' : [10, 50],
    'clf__C' : [1.0, 2.0]

}

In [217]:
cv2 = StratifiedKFold(n_splits=3 ,shuffle=True, random_state=42)

In [218]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=grid_params2,
    cv = cv2,
    scoring='f1_macro',
    verbose=1
)

In [219]:
grid.fit(X_train, Y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


,estimator,Pipeline(step..._iter=2000))])
,param_grid,"{'clf__C': [1.0, 2.0], 'ft__epochs': [10, 50], 'ft__sg': [0, 1], 'ft__window': [3, 5]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,tokenize,<function tok...0026B66715940>


In [220]:
print('Best 파라미터', grid.best_params_)

Best 파라미터 {'clf__C': 2.0, 'ft__epochs': 50, 'ft__sg': 1, 'ft__window': 5}


In [221]:
pred = grid.predict(X_test)
print("classification report : ", classification_report(Y_test, pred))

classification report :                precision    recall  f1-score   support

           0       0.68      0.64      0.66        50
           1       0.66      0.70      0.68        50

    accuracy                           0.67       100
   macro avg       0.67      0.67      0.67       100
weighted avg       0.67      0.67      0.67       100



-----

In [234]:
# document 컬럼에서 문자열 좌우의 공백을 제거
df['document'] = df['document'].astype(str)

In [235]:
# map() 함수를 이용해서 원소 각각을 대입하여 변환
df['document'].map(
    lambda x : x.strip()
)

0                                       아 더빙.. 진짜 짜증나네요 목소리
1                         흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
2                                         너무재밓었다그래서보는것을추천한다
3                             교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
4         사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...
                                ...                        
149995                                  인간이 문제지.. 소는 뭔죄인가..
149996                                        평점이 너무 낮아서...
149997                      이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?
149998                          청춘 영화의 최고봉.방황과 우울했던 날들의 자화상
149999                             한국 영화 최초로 수간하는 내용이 담긴 영화
Name: document, Length: 146183, dtype: object

In [236]:
# Series에서 문자열 Method을 사용할 수 있는 형태로 변환하고 문자열 함수 사용
df['document'] = df['document'].str.strip()

In [237]:
# document에서 중복된 데이터를 제거
df.drop_duplicates('document', inplace=True)

In [238]:
# train, test 데이터를 나눠준다.
x_train = df['document'].head(500).values
y_train = df['label'].head(500).values
x_test = df['document'].tail(100).values
y_test = df['label'].tail(100).values

In [239]:
# 파이프라인 생성
pipe2 = Pipeline(
    [
        ('ft', FastTExtVectorizer(tokenize=tokenize)),
        ('clf', LogisticRegression(max_iter=2000))
    ]
)

In [240]:
cv2 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [244]:
grid_params2 = {
    'ft__window' : [3, 5],
    'ft__sg' : [0, 1],
    'ft__epochs' : [10, 50],
    'clf__C' : [1.0, 2.0]
}

In [245]:
grid2 = GridSearchCV(
    estimator=pipe2,
    cv = cv2,
    param_grid=grid_params2,
    scoring='f1_macro',
    verbose=1
)

In [246]:
grid2.fit(x_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


,estimator,Pipeline(step..._iter=2000))])
,param_grid,"{'clf__C': [1.0, 2.0], 'ft__epochs': [10, 50], 'ft__sg': [0, 1], 'ft__window': [3, 5]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,tokenize,<function tok...0026B66715940>
